In [1]:
from pathlib import Path
import glob, json, math, time
import numpy as np
import pandas as pd

from sklearn.metrics import (
    precision_recall_fscore_support,
    accuracy_score,
    f1_score,
    cohen_kappa_score,
    mean_absolute_error,
    mean_squared_error,
)
from sklearn.preprocessing import MultiLabelBinarizer

# optional correlations
try:
    from scipy.stats import spearmanr, pearsonr
    HAS_SCIPY = True
except Exception:
    HAS_SCIPY = False

# optional display in notebooks
try:
    from IPython.display import display
    HAS_DISPLAY = True
except Exception:
    HAS_DISPLAY = False

import torch
from transformers import AutoTokenizer, AutoModel

In [2]:
GOLD_PATH = Path("ViMUNCH/vimunch_test.json")   # đúng theo folder bạn chụp
RESULT_ROOT = Path("Result_Model")
OUT_DIR = Path("Evaluation")
OUT_DIR.mkdir(parents=True, exist_ok=True)

patterns = [
    "test__zero_shot*.jsonl",
    "test__few_shot*.jsonl",
    "test__ft_*.jsonl",
]
result_files = []
for pat in patterns:
    result_files += list(RESULT_ROOT.rglob(pat))
result_files = sorted([str(p) for p in result_files if Path(p).is_file()])

print("Gold:", GOLD_PATH, "| exists:", GOLD_PATH.exists())
print("Found result files:", len(result_files))
for p in result_files[:20]:
    print("-", p)
if len(result_files) > 20:
    print("... (showing first 20)")

if not GOLD_PATH.exists():
    raise FileNotFoundError(f"Gold file not found: {GOLD_PATH}")
if not result_files:
    raise FileNotFoundError("No result jsonl files found under /mnt/data (patterns: test_zero_shot*, test_few_shot*).")


Gold: ViMUNCH/vimunch_test.json | exists: True
Found result files: 28
- Result_Model/Aya-Expanse-8B/test/test__few_shot_5.jsonl
- Result_Model/Aya-Expanse-8B/test/test__ft_few_shot_5.jsonl
- Result_Model/Aya-Expanse-8B/test/test__ft_zero_shot.jsonl
- Result_Model/Aya-Expanse-8B/test/test__zero_shot.jsonl
- Result_Model/Llama-2-7B/test/test__few_shot_5.jsonl
- Result_Model/Llama-2-7B/test/test__ft_few_shot_5.jsonl
- Result_Model/Llama-2-7B/test/test__ft_zero_shot.jsonl
- Result_Model/Llama-2-7B/test/test__zero_shot.jsonl
- Result_Model/Llama-3.1-8B/test/test__few_shot_5.jsonl
- Result_Model/Llama-3.1-8B/test/test__ft_few_shot_5.jsonl
- Result_Model/Llama-3.1-8B/test/test__ft_zero_shot.jsonl
- Result_Model/Llama-3.1-8B/test/test__zero_shot.jsonl
- Result_Model/Qwen2.5-7B/test/test__few_shot_5.jsonl
- Result_Model/Qwen2.5-7B/test/test__ft_few_shot_5.jsonl
- Result_Model/Qwen2.5-7B/test/test__ft_zero_shot.jsonl
- Result_Model/Qwen2.5-7B/test/test__zero_shot.jsonl
- Result_Model/SeALLMs-v3-

In [3]:
# -------------------------
# 1) LOADERS
# -------------------------
def load_gold(path: Path):
    data = json.loads(path.read_text(encoding="utf-8"))
    return {int(x["id"]): x for x in data}

def iter_jsonl(path: str):
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            yield json.loads(line)

GOLD_BY_ID = load_gold(GOLD_PATH)
print("Gold samples:", len(GOLD_BY_ID))

Gold samples: 1701


In [4]:
# -------------------------
# 2) HELPERS
# -------------------------
def safe_div(a, b):
    return float(a) / float(b) if b else 0.0

def safe_spearman(x, y):
    if len(x) < 2: return np.nan
    if np.std(x) == 0 or np.std(y) == 0: return np.nan
    if HAS_SCIPY:
        return float(spearmanr(x, y).correlation)
    # fallback: rank then pearson
    rx = pd.Series(x).rank(method="average").to_numpy()
    ry = pd.Series(y).rank(method="average").to_numpy()
    if np.std(rx) == 0 or np.std(ry) == 0: return np.nan
    return float(np.corrcoef(rx, ry)[0, 1])

def safe_pearson(x, y):
    if len(x) < 2: return np.nan
    if np.std(x) == 0 or np.std(y) == 0: return np.nan
    if HAS_SCIPY:
        return float(pearsonr(x, y)[0])
    return float(np.corrcoef(x, y)[0, 1])

In [5]:
# -------------------------
# 3) TASK 1B: SPAN METRICS (char-level overlap)
# -------------------------
def normalize_spans(spans, sent_len):
    out = []
    if not spans:
        return out
    for s in spans:
        if not isinstance(s, dict):
            continue
        a = s.get("start", None)
        b = s.get("end", None)
        if a is None or b is None:
            continue
        try:
            a = int(a); b = int(b)
        except Exception:
            continue
        a = max(0, min(a, sent_len))
        b = max(0, min(b, sent_len))
        if b <= a:
            continue
        out.append((a, b))
    out.sort()
    merged = []
    for a, b in out:
        if not merged or a > merged[-1][1]:
            merged.append([a, b])
        else:
            merged[-1][1] = max(merged[-1][1], b)
    return [tuple(x) for x in merged]

def char_set_from_spans(spans, sent_len):
    spans = normalize_spans(spans, sent_len)
    chars = set()
    for a, b in spans:
        chars.update(range(a, b))
    return chars, spans

def span_em(gold_spans, pred_spans, sent_len):
    g = normalize_spans(gold_spans, sent_len)
    p = normalize_spans(pred_spans, sent_len)
    return 1.0 if g == p else 0.0

def span_overlap_metrics(gold_spans, pred_spans, sent_len):
    gset, _ = char_set_from_spans(gold_spans, sent_len)
    pset, _ = char_set_from_spans(pred_spans, sent_len)
    inter = len(gset & pset)
    prec = safe_div(inter, len(pset))
    rec  = safe_div(inter, len(gset))
    f1 = safe_div(2 * prec * rec, (prec + rec)) if (prec + rec) else 0.0
    iou = safe_div(inter, len(gset | pset))
    return prec, rec, f1, iou

In [6]:
# -------------------------
# 4) TASK 3: ROUGE-L / chrF (no extra deps)
# -------------------------
def lcs_len(a_tokens, b_tokens):
    n, m = len(a_tokens), len(b_tokens)
    dp = [0] * (m + 1)
    for i in range(1, n + 1):
        prev = 0
        for j in range(1, m + 1):
            tmp = dp[j]
            if a_tokens[i - 1] == b_tokens[j - 1]:
                dp[j] = prev + 1
            else:
                dp[j] = max(dp[j], dp[j - 1])
            prev = tmp
    return dp[m]

def rouge_l_f1(hyp, ref):
    hyp = (hyp or "").strip()
    ref = (ref or "").strip()
    if not hyp and not ref:
        return 1.0
    if not hyp or not ref:
        return 0.0
    ht = hyp.split()
    rt = ref.split()
    lcs = lcs_len(ht, rt)
    prec = safe_div(lcs, len(ht))
    rec  = safe_div(lcs, len(rt))
    return safe_div(2 * prec * rec, (prec + rec)) if (prec + rec) else 0.0

from collections import Counter

def char_ngrams(s, n):
    s = s.replace(" ", "")
    if len(s) < n:
        return []
    return [s[i:i + n] for i in range(len(s) - n + 1)]

def chrf_score(hyp, ref, n_max=6, beta=2.0):
    hyp = (hyp or "").strip()
    ref = (ref or "").strip()
    if not hyp and not ref:
        return 1.0
    if not hyp or not ref:
        return 0.0

    scores = []
    for n in range(1, n_max + 1):
        h_ngrams = Counter(char_ngrams(hyp, n))
        r_ngrams = Counter(char_ngrams(ref, n))
        if not h_ngrams and not r_ngrams:
            scores.append(1.0); continue
        if not h_ngrams or not r_ngrams:
            scores.append(0.0); continue
        overlap = sum((h_ngrams & r_ngrams).values())
        prec = safe_div(overlap, sum(h_ngrams.values()))
        rec  = safe_div(overlap, sum(r_ngrams.values()))
        if prec == 0 and rec == 0:
            scores.append(0.0)
        else:
            b2 = beta * beta
            scores.append((1 + b2) * prec * rec / (b2 * prec + rec))
    return float(np.mean(scores))

In [7]:
# -------------------------
# 4B) TASK 3: Embedding Cosine Similarity (SimCSE PhoBERT)
# -------------------------
SIMCSE_MODEL_NAME = "VoVanPhuc/sup-SimCSE-VietNamese-phobert-base"

def _get_simcse_model():
    if not hasattr(_get_simcse_model, "_loaded"):
        device = "cuda" if torch.cuda.is_available() else "cpu"
        tok = AutoTokenizer.from_pretrained(SIMCSE_MODEL_NAME)
        mdl = AutoModel.from_pretrained(SIMCSE_MODEL_NAME).to(device)
        mdl.eval()
        _get_simcse_model._loaded = (tok, mdl, device)
    return _get_simcse_model._loaded

def _mean_pool(last_hidden, attention_mask):
    # last_hidden: [B, T, H], attention_mask: [B, T]
    mask = attention_mask.unsqueeze(-1).type_as(last_hidden)  # [B,T,1]
    summed = (last_hidden * mask).sum(dim=1)                  # [B,H]
    denom = mask.sum(dim=1).clamp(min=1e-9)                   # [B,1]
    return summed / denom

@torch.no_grad()
def embed_texts(texts, max_length=256):
    tok, mdl, device = _get_simcse_model()
    enc = tok(
        texts,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt",
    )
    enc = {k: v.to(device) for k, v in enc.items()}
    out = mdl(**enc)
    emb = _mean_pool(out.last_hidden_state, enc["attention_mask"])
    emb = torch.nn.functional.normalize(emb, p=2, dim=1)  # normalize for cosine
    return emb

@torch.no_grad()
def embedding_cosine_similarity(hyp: str, ref: str, max_length=256) -> float:
    hyp = (hyp or "").strip()
    ref = (ref or "").strip()
    if not hyp and not ref:
        return 1.0
    if not hyp or not ref:
        return 0.0
    embs = embed_texts([hyp, ref], max_length=max_length)  # [2,H]
    cos = (embs[0] * embs[1]).sum().item()
    return float(cos)

In [8]:
# -------------------------
# 5) TASK 4: SCALE + ORDINAL
# -------------------------
COMP_QUALITY = ["accuracy", "clarity", "naturalness"]
COMP_SIM = ["meaning", "modality", "implication", "syntax", "context"]
COMP_ALL = COMP_QUALITY + COMP_SIM

def infer_scale_and_convert(scores: dict, target_max=4.0):
    """
    Convert judge scores to 0..4 if they look like 0..1.
    Heuristic: if max(value) <= 1.0 => multiply by 4.
    """
    if not scores or not isinstance(scores, dict):
        return None
    vals = [v for v in scores.values() if isinstance(v, (int, float))]
    if not vals:
        return None
    mx = max(vals)
    factor = target_max if mx <= 1.0000001 else 1.0
    return {k: (float(v) * factor if isinstance(v, (int, float)) else None) for k, v in scores.items()}

def round_clip_0_4(x):
    if x is None: return None
    try:
        v = int(round(float(x)))
    except Exception:
        return None
    return max(0, min(4, v))

In [9]:
# -------------------------
# 6) MODEL/SETTING FROM PATH
# -------------------------
# --- Cell 8 ---
def infer_model_setting_from_path(path: str):
    p = Path(path)
    name = p.name.lower()

    # 1. Xác định Setting cơ bản
    if "few_shot" in name:
        mode = "few_shot"
    elif "zero_shot" in name:
        mode = "zero_shot"
    else:
        mode = "unknown"

    # 2. Kiểm tra xem có phải Fine-Tune không?

    if "_ft_" in name:
        setting = f"ft_{mode}"
    else:
        setting = mode

    # 3. Lấy tên Model (Logic giữ nguyên)
    parent = p.parent
    if parent.name.lower() == "test":
        model = parent.parent.name
    else:
        model = parent.name

    return model, setting

In [10]:
# -------------------------
# 7) EVALUATE ONE FILE (UPDATED)
# -------------------------
def evaluate_file(path: str, gold_by_id: dict):
    model_name, setting = infer_model_setting_from_path(path)

    # Cache label classes once
    if not hasattr(evaluate_file, "_label_classes"):
        label_set = set()
        for g in gold_by_id.values():
            for t in (g.get("metaphor_types") or []):
                t = str(t).strip()
                if t:
                    label_set.add(t)
        evaluate_file._gold_label_set = set(sorted(label_set))
        evaluate_file._label_classes = sorted(label_set) + ["__INVALID__"]
    GOLD_LABEL_SET = evaluate_file._gold_label_set
    LABEL_CLASSES = evaluate_file._label_classes

    # Task 1A
    y_true_det, y_pred_det = [], []

    # Task 1B (gold positives only) + FP spans on negatives
    span_f1_list, span_em_list, span_iou_list = [], [], []
    span_fp_on_neg, neg_count, pos_count = 0, 0, 0

    # Task 2 (gold positives only)
    gold_labels, pred_labels = [], []

    # Task 3 (gold positives only) - semantic focus
    embcos_list = []
    rougeL_list, chrf_list = [], []
    n_t3 = 0

    # Task 4 (gold positives only)
    gold_comp = {k: [] for k in COMP_ALL}
    pred_comp = {k: [] for k in COMP_ALL}
    gold_overall, pred_overall = [], []
    gold_quality, pred_quality = [], []

    n_samples = 0

    for row in iter_jsonl(path):
        n_samples += 1
        sid = int(row.get("id"))
        sent = (row.get("sentence") or "").strip()

        # Support both formats
        gold = row.get("gold") if isinstance(row.get("gold"), dict) else None
        pred = row.get("pred") if isinstance(row.get("pred"), dict) else None
        judge = row.get("judge") if isinstance(row.get("judge"), dict) else None

        if gold is None:
            gold = gold_by_id.get(sid, {})
        if pred is None:
            pred = row

        # ---------- Task 1A: Detection ----------
        gt_det = int(gold.get("have_metaphor", 0))
        pr_det = pred.get("have_metaphor", 0)
        try:
            pr_det = int(pr_det)
        except Exception:
            pr_det = 0

        y_true_det.append(gt_det)
        y_pred_det.append(pr_det)

        # Spans
        g_spans = gold.get("metaphor_phrases") or []
        p_spans = pred.get("metaphor_phrases") or []

        if gt_det == 1:
            pos_count += 1
            # Task 1B: Span Metrics
            _, _, f1s, iou = span_overlap_metrics(g_spans, p_spans, len(sent))
            em = span_em(g_spans, p_spans, len(sent))
            span_f1_list.append(f1s)    # Primary for 1B
            span_iou_list.append(iou)   # Secondary
            span_em_list.append(em)     # Secondary

            # ---------- Task 2 (only on positives) ----------
            g_types = gold.get("metaphor_types") or []
            p_types = pred.get("metaphor_types") or []
            if not isinstance(g_types, list): g_types = []
            if not isinstance(p_types, list): p_types = []

            g_clean = []
            for x in g_types:
                x = str(x).strip()
                if x and (x in GOLD_LABEL_SET):
                    g_clean.append(x)
            g_clean = sorted(set(g_clean))

            p_clean = []
            for x in p_types:
                x = str(x).strip()
                if not x: continue
                if x in GOLD_LABEL_SET:
                    p_clean.append(x)
                else:
                    p_clean.append("__INVALID__")
            p_clean = sorted(set(p_clean))

            gold_labels.append(g_clean)
            pred_labels.append(p_clean)

            # ---------- Task 3 (only on positives) ----------
            ref = (gold.get("interpretation") or "").strip()
            hyp = (pred.get("interpretation") or "").strip()
            if ref:
                # Primary: Embedding Cosine
                embcos_list.append(embedding_cosine_similarity(hyp, ref))
                # Secondary: Text Overlap
                rougeL_list.append(rouge_l_f1(hyp, ref))
                chrf_list.append(chrf_score(hyp, ref))
                n_t3 += 1

            # ---------- Task 4 (judge) - only positives ----------
            g_scores = gold.get("scores") or {}
            j_scores = {}
            if isinstance(judge, dict):
                j_scores = judge.get("scores") or {}
            j_scaled = infer_scale_and_convert(j_scores, target_max=4.0) or {}

            # Components (Ordinal 0..4) -> For QWK (Task 4A)
            for k in COMP_ALL:
                gv = g_scores.get(k, None)
                pv = j_scaled.get(k, None)
                if isinstance(gv, (int, float)) and isinstance(pv, (int, float)):
                    gold_comp[k].append(int(gv))
                    pred_comp[k].append(round_clip_0_4(pv))

            # Overall & Quality (Continuous) -> For MAE (Task 4B)
            # Recompute weighted scores to ensure consistency
            go = g_scores.get("overall", None)
            gq = g_scores.get("quality", None)

            # Calculate Pred Overall from components
            need_overall = ["meaning", "implication", "modality", "syntax", "context"]
            pred_overall_vals = {}
            ok_overall = True
            for k in need_overall:
                pv = j_scaled.get(k, None)
                pv = round_clip_0_4(pv) if isinstance(pv, (int, float)) else None
                pred_overall_vals[k] = pv
                if pv is None: ok_overall = False

            if ok_overall:
                po_calc = (
                    pred_overall_vals["meaning"] * 30
                    + pred_overall_vals["implication"] * 25
                    + pred_overall_vals["modality"] * 25
                    + pred_overall_vals["syntax"] * 10
                    + pred_overall_vals["context"] * 10
                ) / 100.0
            else:
                po_calc = None

            # Calculate Pred Quality from components
            need_quality = ["accuracy", "clarity", "naturalness"]
            pred_quality_vals = {}
            ok_quality = True
            for k in need_quality:
                pv = j_scaled.get(k, None)
                pv = round_clip_0_4(pv) if isinstance(pv, (int, float)) else None
                pred_quality_vals[k] = pv
                if pv is None: ok_quality = False

            if ok_quality:
                pq_calc = (
                    3 * pred_quality_vals["accuracy"]
                    + 2 * pred_quality_vals["clarity"]
                    + pred_quality_vals["naturalness"]
                ) / 6.0
            else:
                pq_calc = None

            if isinstance(go, (int, float)) and isinstance(po_calc, (int, float)):
                gold_overall.append(float(go))
                pred_overall.append(float(po_calc))

            if isinstance(gq, (int, float)) and isinstance(pq_calc, (int, float)):
                gold_quality.append(float(gq))
                pred_quality.append(float(pq_calc))

        else:
            # Negative Sample Logic
            neg_count += 1
            if p_spans:
                span_fp_on_neg += 1

    # ---- Task 1A aggregate (Primary: F1) ----
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true_det, y_pred_det, average="binary", pos_label=1, zero_division=0
    )
    acc = accuracy_score(y_true_det, y_pred_det)

    # ---- Task 1B aggregate (Primary: Overlap F1) ----
    t1B = dict(
        span_overlap_f1_mean_pos=float(np.mean(span_f1_list)) if span_f1_list else np.nan,
        span_em_mean_pos=float(np.mean(span_em_list)) if span_em_list else np.nan,
        span_iou_mean_pos=float(np.mean(span_iou_list)) if span_iou_list else np.nan,
        span_fp_rate_on_neg=safe_div(span_fp_on_neg, neg_count),
        n_pos=int(pos_count),
        n_neg=int(neg_count),
    )

    # ---- Task 2 aggregate (Primary: Macro F1) ----
    if len(gold_labels) > 0:
        mlb = MultiLabelBinarizer(classes=LABEL_CLASSES)
        Y_true = mlb.fit_transform(gold_labels)
        Y_pred = mlb.transform(pred_labels)

        t2_micro = f1_score(Y_true, Y_pred, average="micro", zero_division=0)
        t2_macro = f1_score(Y_true, Y_pred, average="macro", zero_division=0)
        t2_subset = accuracy_score(Y_true, Y_pred)
        per_label_f1 = dict(zip(mlb.classes_, f1_score(Y_true, Y_pred, average=None, zero_division=0)))
    else:
        t2_micro = t2_macro = t2_subset = np.nan
        per_label_f1 = {}

    # ---- Task 3 aggregate (Primary: Emb Cosine) ----
    t3 = dict(
        emb_cos_mean=float(np.mean(embcos_list)) if embcos_list else np.nan,
        rougeL_f1_mean=float(np.mean(rougeL_list)) if rougeL_list else np.nan,
        chrf_mean=float(np.mean(chrf_list)) if chrf_list else np.nan,
        n_eval=int(n_t3),
    )

    # ---- Task 4A aggregate (Ordinal -> QWK) ----
    qwk_list, mae_list, acc_pm1_list = [], [], []
    for k in COMP_ALL:
        g = gold_comp[k]
        p = pred_comp[k]
        if len(g) >= 2 and len(p) == len(g):
            qwk_list.append(cohen_kappa_score(g, p, weights="quadratic"))
            mae_list.append(mean_absolute_error(g, p))
            acc_pm1_list.append(float(np.mean([1.0 if abs(gi - pi) <= 1 else 0.0 for gi, pi in zip(g, p)])))

    t4A = dict(
        qwk_avg_components=float(np.mean(qwk_list)) if qwk_list else np.nan,
        mae_avg_components=float(np.mean(mae_list)) if mae_list else np.nan,
        acc_pm1_avg_components=float(np.mean(acc_pm1_list)) if acc_pm1_list else np.nan,
        n_comp_eval=int(len(gold_comp[COMP_ALL[0]])) if COMP_ALL else 0,
    )

    # ---- Task 4B aggregate (Continuous -> MAE) ----
    def rmse(x, y):
        return math.sqrt(mean_squared_error(x, y))

    t4B = dict(
        mae_overall=mean_absolute_error(gold_overall, pred_overall) if len(gold_overall) else np.nan,
        rmse_overall=rmse(gold_overall, pred_overall) if len(gold_overall) else np.nan,
        spearman_overall=safe_spearman(gold_overall, pred_overall) if len(gold_overall) else np.nan,
        pearson_overall=safe_pearson(gold_overall, pred_overall) if len(gold_overall) else np.nan,

        mae_quality=mean_absolute_error(gold_quality, pred_quality) if len(gold_quality) else np.nan,
        rmse_quality=rmse(gold_quality, pred_quality) if len(gold_quality) else np.nan,
        spearman_quality=safe_spearman(gold_quality, pred_quality) if len(gold_quality) else np.nan,
        pearson_quality=safe_pearson(gold_quality, pred_quality) if len(gold_quality) else np.nan,

        n_overall_eval=int(len(gold_overall)),
    )

    return {
        "model": model_name,
        "setting": setting,
        "file": path,
        "n_samples": int(n_samples),

        # Task 1A
        "t1_det_f1": float(f1),         # Primary
        "t1_det_precision": float(prec),
        "t1_det_recall": float(rec),
        "t1_det_accuracy": float(acc),

        # Task 1B
        **{f"t1_{k}": v for k, v in t1B.items()},

        # Task 2
        "t2_macro_f1": float(t2_macro) if not pd.isna(t2_macro) else np.nan, # Primary
        "t2_micro_f1": float(t2_micro) if not pd.isna(t2_micro) else np.nan,
        "t2_subset_acc": float(t2_subset) if not pd.isna(t2_subset) else np.nan,
        "t2_per_label_f1": per_label_f1,

        # Task 3
        **{f"t3_{k}": v for k, v in t3.items()},

        # Task 4
        **{f"t4A_{k}": v for k, v in t4A.items()},
        **{f"t4B_{k}": v for k, v in t4B.items()},
    }

In [11]:
# -------------------------
# 8) RUN ALL (Đã thêm hiển thị tiến độ)
# -------------------------
import time
import pandas as pd

t0 = time.time()
rows = []
total_files = len(result_files)

print(f"🚀 Found {total_files} files. Starting evaluation...\n")

for i, p in enumerate(result_files, 1):
    # Lấy tên file ngắn gọn để in cho đẹp
    short_name = Path(p).name
    print(f"[{i}/{total_files}] Processing: {short_name} ...", end=" ", flush=True)
    
    try:
        # Gọi hàm evaluate_file
        res = evaluate_file(p, GOLD_BY_ID)
        rows.append(res)
        print("✅ Done")
    except Exception as e:
        print(f"❌ Error: {e}")
        continue

# Tạo DataFrame tổng hợp kết quả
summary_all = pd.DataFrame(rows)

print(f"\n✨ Completed! Evaluated {len(summary_all)}/{total_files} files in {time.time()-t0:.2f}s")

# Hiển thị sơ bộ để kiểm tra
if not summary_all.empty:
    summary_brief = summary_all[["model", "setting", "n_samples", "file"]].sort_values(["model", "setting"])
    if 'HAS_DISPLAY' in globals() and HAS_DISPLAY:
        display(summary_brief.head())
    else:
        print("\n--- First 5 rows ---")
        print(summary_brief.head().to_string(index=False))
else:
    print("⚠️ Warning: No files were evaluated successfully.")

🚀 Found 28 files. Starting evaluation...

[1/28] Processing: test__few_shot_5.jsonl ... ✅ Done
[2/28] Processing: test__ft_few_shot_5.jsonl ... ✅ Done
[3/28] Processing: test__ft_zero_shot.jsonl ... ✅ Done
[4/28] Processing: test__zero_shot.jsonl ... ✅ Done
[5/28] Processing: test__few_shot_5.jsonl ... ✅ Done
[6/28] Processing: test__ft_few_shot_5.jsonl ... ✅ Done
[7/28] Processing: test__ft_zero_shot.jsonl ... ✅ Done
[8/28] Processing: test__zero_shot.jsonl ... ✅ Done
[9/28] Processing: test__few_shot_5.jsonl ... ✅ Done
[10/28] Processing: test__ft_few_shot_5.jsonl ... ✅ Done
[11/28] Processing: test__ft_zero_shot.jsonl ... ✅ Done
[12/28] Processing: test__zero_shot.jsonl ... ✅ Done
[13/28] Processing: test__few_shot_5.jsonl ... ✅ Done
[14/28] Processing: test__ft_few_shot_5.jsonl ... ✅ Done
[15/28] Processing: test__ft_zero_shot.jsonl ... ✅ Done
[16/28] Processing: test__zero_shot.jsonl ... ✅ Done
[17/28] Processing: test__few_shot_5.jsonl ... ✅ Done
[18/28] Processing: test__ft_few_

,model,setting,n_samples,file
0,Aya-Expanse-8B,few_shot,1701,Result_Model/Aya-Expanse-8B/test/test__few_sho...
1,Aya-Expanse-8B,ft_few_shot,1701,Result_Model/Aya-Expanse-8B/test/test__ft_few_...
2,Aya-Expanse-8B,ft_zero_shot,1701,Result_Model/Aya-Expanse-8B/test/test__ft_zero...
3,Aya-Expanse-8B,zero_shot,1701,Result_Model/Aya-Expanse-8B/test/test__zero_sh...
4,Llama-2-7B,few_shot,1701,Result_Model/Llama-2-7B/test/test__few_shot_5....


In [12]:
# -------------------------
# 9) BUILD TASK TABLES (ALIGNED WITH DESIGN)
# -------------------------

# === TASK 1A: Detection ===
# Primary: F1 (Positive class)
# Secondary: Precision, Recall, Accuracy
task1A = summary_all[[
    "model", "setting", "n_samples",
    "t1_det_f1",       # PRIMARY (sort desc)
    "t1_det_precision",
    "t1_det_recall",
    "t1_det_accuracy"
]].sort_values(["setting", "t1_det_f1"], ascending=[True, False])

# === TASK 1B: Span Extraction ===
# Primary: Overlap F1
# Secondary: IoU, EM
# Mandatory Check: FP Rate on Negatives
task1B = summary_all[[
    "model", "setting", "n_samples",
    "t1_span_overlap_f1_mean_pos", # PRIMARY (sort desc)
    "t1_span_iou_mean_pos",
    "t1_span_em_mean_pos",
    "t1_span_fp_rate_on_neg",      # Check for hallucination
    "t1_n_pos", "t1_n_neg"
]].sort_values(["setting", "t1_span_overlap_f1_mean_pos"], ascending=[True, False])

# === TASK 2: Metaphor Types ===
# Primary: Macro F1
# Secondary: Micro F1, Subset Acc
task2 = summary_all[[
    "model", "setting", "n_samples",
    "t2_macro_f1",     # PRIMARY (sort desc)
    "t2_micro_f1",
    "t2_subset_acc"
]].sort_values(["setting", "t2_macro_f1"], ascending=[True, False])

# === TASK 3: Interpretation ===
# Primary: Embedding Cosine Similarity (SimCSE)
# Secondary: ROUGE-L, chrF
task3 = summary_all[[
    "model", "setting", "n_samples",
    "t3_emb_cos_mean",    # PRIMARY (sort desc)
    "t3_rougeL_f1_mean",
    "t3_chrf_mean",
    "t3_n_eval"
]].sort_values(["setting", "t3_emb_cos_mean"], ascending=[True, False])

# === TASK 4A: Judgement Components (Ordinal) ===
# Primary: QWK (Quadratic Weighted Kappa)
# Secondary: MAE
task4A = summary_all[[
    "model", "setting", "n_samples",
    "t4A_qwk_avg_components",  # PRIMARY (sort desc)
    "t4A_mae_avg_components",
    "t4A_acc_pm1_avg_components",
    "t4A_n_comp_eval"
]].sort_values(["setting", "t4A_qwk_avg_components"], ascending=[True, False])

# === TASK 4B: Judgement Overall (Continuous) ===
# Primary: MAE
# Secondary: RMSE, Spearman, Pearson
task4B = summary_all[[
    "model", "setting", "n_samples",
    "t4B_mae_overall",      # PRIMARY (sort asc - lower is better)
    "t4B_rmse_overall",
    "t4B_spearman_overall",
    "t4B_pearson_overall",
    "t4B_n_overall_eval"
]].sort_values(["setting", "t4B_mae_overall"], ascending=[True, True])

print("\n=== TASK 1A (Detection - Primary: F1) ===")
print(task1A.to_string(index=False))

print("\n=== TASK 1B (Span - Primary: Overlap F1) ===")
print(task1B.to_string(index=False))

print("\n=== TASK 2 (Types - Primary: Macro F1) ===")
print(task2.to_string(index=False))

print("\n=== TASK 3 (Interpretation - Primary: SimCSE Cosine) ===")
print(task3.to_string(index=False))

print("\n=== TASK 4A (Rubric Components - Primary: QWK) ===")
print(task4A.to_string(index=False))

print("\n=== TASK 4B (Rubric Overall - Primary: MAE) ===")
print(task4B.to_string(index=False))

print(f"\nDone in {time.time()-t0:.2f}s")


=== TASK 1A (Detection - Primary: F1) ===
          model      setting  n_samples  t1_det_f1  t1_det_precision  t1_det_recall  t1_det_accuracy
     Qwen2.5-7B     few_shot       1701   0.677561          0.564990       0.846154         0.698413
  SeALLMs-v3-7B     few_shot       1701   0.654845          0.548832       0.811617         0.679600
   vinallama-7b     few_shot       1701   0.614555          0.467980       0.894819         0.579659
Vistral-7B-Chat     few_shot       1701   0.609086          0.446559       0.957614         0.539683
 Aya-Expanse-8B     few_shot       1701   0.560932          0.392476       0.982732         0.423868
   Llama-3.1-8B     few_shot       1701   0.553995          0.384522       0.990581         0.402704
     Llama-2-7B     few_shot       1701   0.544288          0.374118       0.998430         0.373898
Vistral-7B-Chat  ft_few_shot       1701   0.771845          0.796327       0.748823         0.834215
     Qwen2.5-7B  ft_few_shot       1701   0.7570

In [13]:
# -------------------------
# 10) SAVE RESULTS
# -------------------------
# CSVs
summary_all_flat = summary_all.copy()
# make dict column safe for csv/excel
summary_all_flat["t2_per_label_f1"] = summary_all_flat["t2_per_label_f1"].apply(lambda d: json.dumps(d, ensure_ascii=False))

summary_all_flat.to_csv(OUT_DIR / "eval_summary_all.csv", index=False)
task1A.to_csv(OUT_DIR / "eval_task1A.csv", index=False)
task1B.to_csv(OUT_DIR / "eval_task1B.csv", index=False)
task2.to_csv(OUT_DIR / "eval_task2.csv", index=False)
task3.to_csv(OUT_DIR / "eval_task3.csv", index=False)
task4A.to_csv(OUT_DIR / "eval_task4A.csv", index=False)
task4B.to_csv(OUT_DIR / "eval_task4B.csv", index=False)

# per-label f1 json (Task 2)
per_label = {
    f"{row['model']}|{row['setting']}": row["t2_per_label_f1"]
    for _, row in summary_all.iterrows()
}
(OUT_DIR / "eval_task2_per_label_f1.json").write_text(
    json.dumps(per_label, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

# Excel multi-sheet (optional but handy)
xlsx_path = OUT_DIR / "eval_results.xlsx"
try:
    with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
        summary_all_flat.to_excel(writer, sheet_name="summary_all", index=False)
        task1A.to_excel(writer, sheet_name="task1A_detection", index=False)
        task1B.to_excel(writer, sheet_name="task1B_span", index=False)
        task2.to_excel(writer, sheet_name="task2_types", index=False)
        task3.to_excel(writer, sheet_name="task3_interpret", index=False)
        task4A.to_excel(writer, sheet_name="task4A_components", index=False)
        task4B.to_excel(writer, sheet_name="task4B_overall", index=False)
    excel_saved = True
except Exception as e:
    excel_saved = False
    print("Excel export skipped (openpyxl/permission issue):", e)

print("\n✅ Saved results to:", OUT_DIR)
print("-", OUT_DIR / "eval_summary_all.csv")
print("-", OUT_DIR / "eval_task1A.csv")
print("-", OUT_DIR / "eval_task1B.csv")
print("-", OUT_DIR / "eval_task2.csv")
print("-", OUT_DIR / "eval_task3.csv")
print("-", OUT_DIR / "eval_task4A.csv")
print("-", OUT_DIR / "eval_task4B.csv")
print("-", OUT_DIR / "eval_task2_per_label_f1.json")
if excel_saved:
    print("-", xlsx_path)

Excel export skipped (openpyxl/permission issue): No module named 'openpyxl'

✅ Saved results to: Evaluation
- Evaluation/eval_summary_all.csv
- Evaluation/eval_task1A.csv
- Evaluation/eval_task1B.csv
- Evaluation/eval_task2.csv
- Evaluation/eval_task3.csv
- Evaluation/eval_task4A.csv
- Evaluation/eval_task4B.csv
- Evaluation/eval_task2_per_label_f1.json
